# Fase 1: Ingesta y Tratamiento Inicial de Datos
## Premier League Match Recommender

Este notebook documenta la **Fase 1 del proyecto**, enfocada en:
1. Conexión e ingesta de fuentes de datos (Estrategia Zero-Cost Data con fallback local).
2. Exploración y tratamiento de datos de los 20 clubes de la Premier League (estadísticas ofensivas $xG$, defensivas $xGA$, puntos y Star Power ICT).
3. Estructuración y cruce de datos relacionales del calendario de partidos (Fixtures, derbis y emparejamientos).
4. Ingesta y validación de narrativas textuales cualitativas (prensa y foros de afición).
5. Validación estricta de esquemas de datos con Pydantic (`Team`, `Fixture`).

In [1]:
import sys
import os
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Configurar ruta base del proyecto
BASE_DIR = Path(os.path.abspath("..")) if Path("..").resolve().name == "sistema_de_recomendaciones" else Path(os.path.abspath("."))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.core.config import settings
from src.core.schemas import Team, Fixture
from src.data_ingestion.local_provider import LocalDataProvider
from src.data_ingestion.fpl_client import FPLDataProvider

print("✔ Entorno configurado correctamente.")
print(f"✔ Directorio de datos: {settings.data_dir}")
print(f"✔ Versión de Pandas: {pd.__version__}")

✔ Entorno configurado correctamente.
✔ Directorio de datos: /Users/hugoportillo/sistema_de_recomendaciones/data
✔ Versión de Pandas: 2.3.3


---
## 1. Tratamiento y Análisis Exploratorio de Clubes (`Teams`)

Cargamos y estructuramos los datos de los 20 equipos de la Premier League desde `sample_teams.json` validándolos contra el esquema Pydantic `Team`.
Posteriormente, construimos un DataFrame enriquecido con métricas analíticas calculadas.

In [2]:
local_provider = LocalDataProvider(data_dir=settings.data_dir)
teams = local_provider.get_teams()

# Convertir a DataFrame para análisis y tratamiento
teams_data = [
    {
        "id": t.id,
        "name": t.name,
        "short_name": t.short_name,
        "position": t.position,
        "points": t.points,
        "played": t.played,
        "xg_last5": t.xg_for_last5,
        "xga_last5": t.xga_for_last5,
        "form": t.form,
        "star_power_ict": t.key_players_ict,
        "rivals_count": len(t.rivals),
        "rivals_list": ", ".join(t.rivals) if t.rivals else "Ninguno"
    }
    for t in teams
]

df_teams = pd.DataFrame(teams_data).sort_values("position").reset_index(drop=True)

# Cálculo de variables derivadas
df_teams["points_per_game"] = round(df_teams["points"] / df_teams["played"], 2)
df_teams["xg_diff_last5"] = round(df_teams["xg_last5"] - df_teams["xga_last5"], 2)

# Segmentación por zona competitiva en la tabla
def clasificar_zona(pos):
    if pos <= 4:
        return "Zona Champions / Título"
    elif pos <= 7:
        return "Zona Europa"
    elif pos <= 15:
        return "Media Tabla"
    else:
        return "Zona Descenso"

df_teams["tier"] = df_teams["position"].apply(clasificar_zona)

print(f"Total de clubes cargados y validados: {len(df_teams)}")
print(df_teams[["position", "name", "points", "points_per_game", "xg_last5", "xga_last5", "xg_diff_last5", "star_power_ict", "tier"]].to_string(index=False))

Total de clubes cargados y validados: 20
 position                    name  points  points_per_game  xg_last5  xga_last5  xg_diff_last5  star_power_ict                    tier
        1                 Arsenal      64             2.29      2.45       0.65           1.80           310.5 Zona Champions / Título
        2               Liverpool      64             2.29      2.60       1.10           1.50           325.0 Zona Champions / Título
        3         Manchester City      63             2.25      2.30       0.85           1.45           340.0 Zona Champions / Título
        4             Aston Villa      55             1.96      1.95       1.40           0.55           240.0 Zona Champions / Título
        5       Tottenham Hotspur      53             1.89      2.10       1.65           0.45           260.0             Zona Europa
        6       Manchester United      47             1.68      1.45       1.70          -0.25           230.0             Zona Europa
        7     

### Estadísticas Descriptivas y Perfil de Rendimiento

A continuación se analizan los rangos de dispersión, medias y desviaciones de las variables de rendimiento ofensivo ($xG$), defensivo ($xGA$) y el Star Power ICT.

In [3]:
stats_summary = df_teams[["points", "xg_last5", "xga_last5", "xg_diff_last5", "star_power_ict"]].describe()
print("--- RESUMEN ESTADÍSTICO DE LOS 20 EQUIPOS ---")
print(stats_summary.round(2).to_string())

# Identificar Top 3 mejores ataques y defensas por xG
top_ataque = df_teams.sort_values("xg_last5", ascending=False)[["name", "xg_last5", "position"]].head(3)
top_defensa = df_teams.sort_values("xga_last5", ascending=True)[["name", "xga_last5", "position"]].head(3)

print("\n🔥 Top 3 Ataques más Potentes (xG últimos 5 partidos):")
for _, r in top_ataque.iterrows():
    print(f"   • {r['name']} (#{r['position']}): {r['xg_last5']} xG/partido")

print("\n🛡️ Top 3 Defensas más Sólidas (xGA últimos 5 partidos):")
for _, r in top_defensa.iterrows():
    print(f"   • {r['name']} (#{r['position']}): {r['xga_last5']} xGA/partido")

--- RESUMEN ESTADÍSTICO DE LOS 20 EQUIPOS ---
       points  xg_last5  xga_last5  xg_diff_last5  star_power_ict
count   20.00     20.00      20.00          20.00           20.00
mean    38.55      1.62       1.56           0.06          198.28
std     15.72      0.50       0.44           0.85           68.93
min     14.00      0.85       0.65          -1.65          110.00
25%     25.75      1.27       1.34          -0.40          148.75
50%     39.50      1.45       1.52           0.05          180.00
75%     48.50      1.98       1.75           0.45          232.50
max     64.00      2.60       2.50           1.80          340.00

🔥 Top 3 Ataques más Potentes (xG últimos 5 partidos):
   • Liverpool (#2): 2.6 xG/partido
   • Arsenal (#1): 2.45 xG/partido
   • Manchester City (#3): 2.3 xG/partido

🛡️ Top 3 Defensas más Sólidas (xGA últimos 5 partidos):
   • Arsenal (#1): 0.65 xGA/partido
   • Manchester City (#3): 0.85 xGA/partido
   • Liverpool (#2): 1.1 xGA/partido


---
## 2. Tratamiento y Cruce del Calendario de Partidos (`Fixtures`)

Examinamos los partidos programados para la Jornada 29. Realizamos un cruce de datos (*merge / join*) con las métricas de los clubes locales y visitantes para generar una vista consolidada de los encuentros.

In [4]:
fixtures = local_provider.get_fixtures(gameweek=29)

fixtures_data = []
for f in fixtures:
    h = f.home_team
    a = f.away_team
    fixtures_data.append({
        "fixture_id": f.id,
        "gameweek": f.gameweek,
        "kickoff_time": f.kickoff_time,
        "home_id": h.id,
        "home_team": h.name,
        "home_pos": h.position,
        "home_xg": h.xg_for_last5,
        "home_ict": h.key_players_ict,
        "away_id": a.id,
        "away_team": a.name,
        "away_pos": a.position,
        "away_xg": a.xg_for_last5,
        "away_ict": a.key_players_ict,
        "is_derby": f.is_derby,
        "derby_name": f.derby_name or "Partido Regular",
        "pos_diff": abs(h.position - a.position)
    })

df_fixtures = pd.DataFrame(fixtures_data)
print(f"Total de partidos en la Jornada 29: {len(df_fixtures)}")
print(df_fixtures[["fixture_id", "home_team", "away_team", "home_pos", "away_pos", "pos_diff", "is_derby", "derby_name"]].to_string(index=False))

Total de partidos en la Jornada 29: 10
 fixture_id              home_team               away_team  home_pos  away_pos  pos_diff  is_derby             derby_name
        101        Manchester City                 Arsenal         3         1         2     False    Title Decider Clash
        102              Liverpool                 Everton         2        16        14      True       Merseyside Derby
        103      Tottenham Hotspur                 Chelsea         5        11         6      True           London Derby
        104       Newcastle United         West Ham United        10         7         3     False        Partido Regular
        105            Aston Villa Wolverhampton Wanderers         4         9         5      True    West Midlands Derby
        106             Luton Town       Nottingham Forest        18        17         1     False Relegation Six-Pointer
        107 Brighton & Hove Albion          Crystal Palace         8        14         6      True        A

### Detección y Análisis de Derbis de la Jornada

Los partidos catalogados como derbis o con alta rivalidad histórica reciben una consideración especial en el modelo. Evaluamos la presencia de clásicos locales en la jornada.

In [5]:
derbis_jornada = df_fixtures[df_fixtures["is_derby"] == True]
print(f"Derbis identificados en la Jornada 29 ({len(derbis_jornada)} encuentros):")
for _, d in derbis_jornada.iterrows():
    print(f" ⚔️  {d['derby_name']}: {d['home_team']} (#{d['home_pos']}) vs {d['away_team']} (#{d['away_pos']}) [Kickoff: {d['kickoff_time']}]")

Derbis identificados en la Jornada 29 (4 encuentros):
 ⚔️  Merseyside Derby: Liverpool (#2) vs Everton (#16) [Kickoff: 2026-03-30T11:30:00Z]
 ⚔️  London Derby: Tottenham Hotspur (#5) vs Chelsea (#11) [Kickoff: 2026-03-30T16:30:00Z]
 ⚔️  West Midlands Derby: Aston Villa (#4) vs Wolverhampton Wanderers (#9) [Kickoff: 2026-03-30T14:00:00Z]
 ⚔️  A23 / M23 Derby: Brighton & Hove Albion (#8) vs Crystal Palace (#14) [Kickoff: 2026-03-31T13:00:00Z]


---
## 3. Ingesta Externa con Fallback Automático (`FPLDataProvider`)

El sistema utiliza la **Estrategia Zero-Cost Data**:
- Intenta consumir la API oficial y pública de Fantasy Premier League (`/bootstrap-static/`).
- Si se detecta tiempo de espera o ausencia de conectividad, realiza un *fallback* transparente al proveedor local (`LocalDataProvider`).

In [6]:
fpl_provider = FPLDataProvider(fallback_provider=local_provider)
fpl_teams = fpl_provider.get_teams()
fpl_fixtures = fpl_provider.get_fixtures(gameweek=29)

print(f"Proveedor FPL activo. Equipos extraídos: {len(fpl_teams)} | Partidos jornada 29: {len(fpl_fixtures)}")
print(f"Ejemplo de equipo extraído vía proveedor: {fpl_teams[0].name} (ICT: {fpl_teams[0].key_players_ict})")

Proveedor FPL activo. Equipos extraídos: 20 | Partidos jornada 29: 10
Ejemplo de equipo extraído vía proveedor: Arsenal (ICT: 310.5)


---
## 4. Ingesta y Tratamiento de Narrativas Cualitativas Semilla

Para cada partido, el sistema extrae fuentes textuales que alimentarán el pipeline de Procesamiento de Lenguaje Natural (PLN):
- Titulares de prensa deportiva (`headlines`).
- Comentarios de aficionados en comunidades de Reddit (`fan_comments`).
- Volumen de cobertura en medios (`media_articles_count`).

In [7]:
narratives_summary = []
for f in fixtures:
    narrative = local_provider.get_narratives(f.id)
    headlines = narrative.get("headlines", [])
    comments = narrative.get("fan_comments", [])
    media_count = narrative.get("media_articles_count", 0)
    
    narratives_summary.append({
        "fixture_id": f.id,
        "match": f"{f.home_team.name} vs {f.away_team.name}",
        "headlines_count": len(headlines),
        "comments_count": len(comments),
        "media_articles_volume": media_count,
        "sample_headline": headlines[0] if headlines else "N/A"
    })

df_narratives = pd.DataFrame(narratives_summary)
print("--- RESUMEN DE COBERTURA CUALITATIVA POR ENCUENTRO ---")
print(df_narratives[["fixture_id", "match", "media_articles_volume", "sample_headline"]].to_string(index=False))

--- RESUMEN DE COBERTURA CUALITATIVA POR ENCUENTRO ---
 fixture_id                                    match  media_articles_volume                                                                           sample_headline
        101               Manchester City vs Arsenal                    142    Manchester City vs Arsenal: The monumental Premier League title showdown at the Etihad
        102                     Liverpool vs Everton                     98                Merseyside Derby: Liverpool host Everton in fiery local rivalry at Anfield
        103             Tottenham Hotspur vs Chelsea                     89       Tottenham vs Chelsea: Fierce London derby packed with bad blood and attacking chaos
        104      Newcastle United vs West Ham United                     52             Newcastle United face West Ham in thrilling battle for European qualification
        105   Aston Villa vs Wolverhampton Wanderers                     48   Aston Villa vs Wolves: Intense West M

---
## 5. Validación y Consolidación de Datos para las Siguientes Fases

Verificamos la consistencia de los identificadores entre todas las tablas y generamos la estructura limpia de entrada para el pipeline de extracción de características cuantitativas y cualitativas.

In [8]:
# Validación de integridad referencial
assert len(df_teams) == 20, "Deben existir exactamente 20 clubes"
assert len(df_fixtures) == 10, "Deben existir exactamente 10 partidos por jornada"
assert set(df_fixtures["fixture_id"]).issubset(set(df_narratives["fixture_id"])), "Todos los fixtures deben tener narrativa asociada"

print("✅ Todos los controles de integridad y validación de esquemas superados con éxito.")
print("✅ Datos preparados y listos para alimentar el Pipeline de PLN (Fase 2) y el Pipeline Numérico (Fase 3).")

✅ Todos los controles de integridad y validación de esquemas superados con éxito.
✅ Datos preparados y listos para alimentar el Pipeline de PLN (Fase 2) y el Pipeline Numérico (Fase 3).
